## Imports

In [113]:
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_ollama import ChatOllama
import os
from dotenv import load_dotenv
import requests
from global_variables import LOCAL_MODEL_NAME, LOCAL_BASE_URL


## Globale Variablen

In [114]:
system_prompt = "Du bist ein Tutor für Vorlesungsinhalte. Beantworte Fragen nur auf Basis des bereitgestellten Kontexts. Erkläre klar, korrekt und verständlich. Wenn Informationen fehlen oder unsicher sind, sage das ausdrücklich. Erfinde nichts und spekuliere nicht. Nutze Fachbegriffe korrekt und erkläre sie kurz, wenn nötig."

LOCAL = True

## Tools

Retreival Tool

In [115]:
# Das Tool stellt Anfragen an die VectorDB und bekommt die entsprechenden Chunks zurück
@tool('search_lecture_docs', description='Retrieves information from Lecture related Documents')
def search_lecture_docs(query: str):
    response = requests.post(
    "http://127.0.0.1:8000/query",
    json={
        "query": query,
        "n": 3
    })
    return response.json()

## Initialisierungen

In [116]:
# Model für Lokale Ollama Modelle
model_local = ChatOllama(
    base_url=LOCAL_BASE_URL,
    model=LOCAL_MODEL_NAME,
)
# Model für Nvidia NIM API
load_dotenv()
model_api = ChatOpenAI(
    model="meta/llama-3.1-8b-instruct",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"],
)

agent = create_agent(model_local if LOCAL else model_api, tools=[search_lecture_docs], system_prompt=system_prompt)

In [117]:
prompt = str(input())
result = agent.invoke({"messages": [("user", prompt)]})
print(result["messages"][-1].content)

Ja – die Organisation unterstützt die fachliche Weiterbildung der Mitarbeitenden.  
Jeder Mitarbeiter kann **jährlich bis zu 1 000 €** für berufliche Weiterbildungsmaßnahmen beantragen. Damit steht eine finanzielle Förderung für Kurse, Seminare, Zertifizierungen oder ähnliche Qualifizierungsangebote zur Verfügung.
